In [1]:
import torch
import math
import pickle
from helper_functions import encode_corpus, load_wikipedia_text, make_dataloaders
from tokenizers import Tokenizer
from transformer_lm import TransformerLM


/zhome/32/4/214716/.local/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
SEQ_LEN = 128
BATCH_SIZE = 64
LEARNING_RATE = 1e-4

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

tokenizers = [
    "bpe",
    # "unigram"
]

for tokenizer_name in tokenizers:

    tokenizer = Tokenizer.from_file(f"tokenizers/{tokenizer_name}_tokenizer.json")

    vocab_size = tokenizer.get_vocab_size()

    text_en = load_wikipedia_text(language="en", target_chars=20000)
    text_ru = load_wikipedia_text(language="ru", target_chars=20000)

    text = text_en + text_ru

    ids = encode_corpus(tokenizer, text)

    _, _, test_loader = make_dataloaders(ids, seq_len=SEQ_LEN, batch_size=BATCH_SIZE)

    model = TransformerLM(vocab_size=vocab_size, max_seq_length=SEQ_LEN).to(device)
    state_dict = torch.load(f"models/{tokenizer_name}_transformer.pth", map_location=device, weights_only=True)
    model.load_state_dict(state_dict)

    history = pickle.load(open(f"history/{tokenizer_name}_training_history.pkl", "rb"))
    criterion = torch.nn.CrossEntropyLoss()

    with torch.no_grad():
        model.eval()
        test_loss = 0.0
        count = 0

        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = criterion(logits.view(-1, vocab_size), y.view(-1))
            test_loss += loss.item()
            count += 1

        avg_test_loss = test_loss / max(count, 1)
        test_ppl = math.exp(avg_test_loss)

        test_bpc = avg_test_loss / math.log(2)

        print(f"Results for model trained with {tokenizer_name}:")
        print(f"Test Loss: {avg_test_loss:.4f}, Test PPL: {test_ppl:.4f}, Test BPC: {test_bpc:.4f}")

Using device: cuda


Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/41 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/21 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/21 [00:00<?, ?it/s]

Encoded 2 texts into 9500 token IDs
Vocabulary size: 16384
Results for model trained with bpe:
Test Loss: 0.0000, Test PPL: 1.0000, Test BPC: 0.0000
